# Tool Use, Function Calling & MCP

Companion notebook for the [Tool Use & MCP lesson](https://ml-viz-ruby.vercel.app/courses/agent-design-patterns/09-tool-use-and-mcp).

**The idea in one sentence.** Agents act by **calling tools** (function calling), and
**MCP** (the Model Context Protocol) standardises how tools are described and invoked —
but the hard truth is that reliability **compounds**: an agent that is 95% reliable per
step is only $0.95^{10}\approx 60\%$ reliable over a 10-step task.

Two facts every agent builder must internalise:

- **Success $= p^n$:** end-to-end reliability is the per-step reliability raised to the
  trajectory length — long tasks need *very* high per-step reliability.
- **Retries rescue flaky steps:** a step that fails 60% of the time becomes reliable
  with a few retries ($1-f^k$).

We build a toy tool-calling loop and **validate the compounding and retry formulas**,
then cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Dark style matching the site theme.
plt.style.use('dark_background')
plt.rcParams.update({
    'axes.edgecolor': '#475569',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'axes.titlecolor': '#e2e8f0',
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'grid.color': '#2e3347',
    'savefig.facecolor': '#0f1117',
})
BRAND = '#6366f1'
TEAL = '#14b8a6'
ROSE = '#f43f5e'
YELLOW = '#eab308'

rng = np.random.default_rng(0)

## 1. A toy tool-calling loop

The model never runs tools — it *emits a call* that our runtime executes. We mock a model that parses a question into tool calls (`search`, `calc`) and feed observations back. The point is the control flow, not a real LLM.

In [ ]:
TOOLS = {
    'capital': {'France': 'Paris', 'Japan': 'Tokyo'},
    'population': {'Paris': 2_100_000, 'Tokyo': 14_000_000},
}

def run_tool(name, arg):
    """Your runtime executes the tool the 'model' requested."""
    if name == 'calc':
        return eval(arg, {'__builtins__': {}})
    return TOOLS[name].get(arg, 'UNKNOWN')

# A scripted 'agent': a list of (thought, tool, arg) emitted step by step.
trajectory = [
    ('need the capital of France', 'capital', 'France'),
    ('now its population',         'population', 'Paris'),
    ('divide by 1000',            'calc', '2100000 / 1000'),
]
obs = None
for thought, tool, arg in trajectory:
    print(f'Thought: {thought:32s} Action: {tool}({arg!r})')
    obs = run_tool(tool, arg)
    print(f'   Observation: {obs}')
print('Answer:', obs)

## 2. Why reliability is $p^n$

If each Thought→Action→Observation step succeeds with probability $p$, an $n$-step trajectory succeeds with probability $p^n$. We plot it: a 95%-reliable step decays fast.

In [ ]:
n = np.arange(0, 21)
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for p in [0.99, 0.95, 0.90, 0.80]:
    ax.plot(n, p**n * 100, 'o-', lw=2, markersize=4, label=f'p = {p:.2f}')
ax.axhline(50, color=ROSE, ls='--', lw=1, alpha=0.6, label='50%')
ax.set_xlabel('trajectory length n (steps)')
ax.set_ylabel('end-to-end success (%)')
ax.set_title('Agents compound errors: success = p^n')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print('At p=0.95, 10 steps:', round(0.95**10 * 100, 1), '%')

### Validate: reliability compounds as $p^n$

The single most important agent equation. We confirm that stringing together reliable
steps degrades fast: at 95% per step, a 10-step trajectory is already a coin flip, and
at 20 steps it's worse than a third.

In [ ]:
for p in [0.99, 0.95, 0.90]:
    for n in [1, 10, 20]:
        print(f'p={p}, n={n:2d}: end-to-end success = {p**n:.3f}')
    print()
assert abs(0.95**10 - 0.5987) < 1e-3, 'p^n compounding'
assert 0.99**20 > 0.90**5, 'high per-step reliability matters more as trajectories grow'
print('✅ success = p^n — long agent trajectories demand very high per-step reliability')

## 3. Retries recover a flaky step

If a tool fails transiently with probability $f$ per attempt, allowing $k$ attempts makes that step succeed with probability $1-f^{k}$. Three retries turn a 60%-failure step into a reliable one.

In [ ]:
f = 0.6  # per-attempt failure of one flaky tool
ks = np.arange(1, 7)
step_success = 1 - f**ks
for k, s in zip(ks, step_success):
    print(f'{k} attempts: step success = {s:.1%}')

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(ks, step_success * 100, 'o-', color=TEAL, lw=2, markersize=8)
ax.set_xlabel('attempts allowed (k)'); ax.set_ylabel('step success (%)')
ax.set_title('Retries turn a 60%-failure step reliable'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

### Validate: retries turn a flaky step reliable

If a step fails independently with probability $f$ per attempt, $k$ attempts succeed
with probability $1-f^k$. Even a 60%-failure step becomes reliable with a handful of
retries — the cheapest reliability lever, provided the tool is **idempotent**.

In [ ]:
f = 0.6
for k in ks:
    succ = 1 - f**k
    print(f'{k} attempts: step success = {succ:.1%}')
assert abs((1 - f**1) - 0.4) < 1e-9, 'one attempt = 1-f'
assert (1 - f**6) > 0.95, 'a few retries make a 60%-failure step reliable'
assert all((1 - f**ks[i]) < (1 - f**ks[i+1]) for i in range(len(ks)-1)), 'more retries never hurt'
print('\n✅ retries rescue flaky steps (1 - f^k) — but only for idempotent tools')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **compounding errors** | $p^n$ — a "reliable" 95% step is a coin flip over 10 steps |
| **non-idempotent tools** | retrying a "send payment" tool double-charges; make tools safe to retry |
| **no retry budget** | infinite retries mask real failures and burn cost; cap them |
| **silent tool failures** | a tool returning a plausible-but-wrong result poisons the trajectory |
| **schema drift** | tool/MCP schemas change; validate arguments before executing |

Demo: making steps retryable beats even a large raw per-step reliability bump.

In [ ]:
# Combine both: the RIGHT lever depends on where the unreliability is. Raising per-step
# reliability helps a long trajectory far more than retrying one step. Compare a 12-step
# task at p=0.90 vs p=0.97 vs p=0.90-with-3-retries-per-step.
n = 12
base = 0.90 ** n
better_p = 0.97 ** n
retried = (1 - (1 - 0.90) ** 3) ** n     # 3 retries per step, each step now ~0.999
print(f'12 steps @ p=0.90            : {base:.3f}')
print(f'12 steps @ p=0.97            : {better_p:.3f}')
print(f'12 steps @ p=0.90 + 3 retries: {retried:.3f}')
assert retried > better_p > base
print('\nRetries (making each step ~0.999) beat even a big raw-reliability bump -> design for retryable steps.')

## ✏️ Your turn — compounded reliability

Implement `trajectory_reliability(p, n)` = the probability an $n$-step agent succeeds end-to-end when each step is `p` reliable.

In [ ]:
def trajectory_reliability(p, n):
    """TODO(you): return p**n."""
    # TODO
    return ...


In [ ]:
r10 = trajectory_reliability(0.95, 10)
r1 = trajectory_reliability(0.99, 1)
print('p=0.95, n=10 ->', round(r10, 4), '(expected ~0.5987)')
assert abs(r10 - 0.95**10) < 1e-9
assert abs(r1 - 0.99) < 1e-9
assert abs(trajectory_reliability(1.0, 100) - 1.0) < 1e-9
print('\n✅ reliability compounds as expected.')

<details>
<summary>Solution</summary>

```python
def trajectory_reliability(p, n):
    return p ** n
```

The takeaway: keep $n$ small (bounded loops, fewer steps), push $p$ up (validated tool I/O, retries on transient failures), and gate the irreversible steps with a human.
</details>

## Recap

- The model **emits** tool calls; your **runtime executes** them and returns observations.
- Reliability is $p^n$ — a 95% step is only ~60% over ten steps.
- Retries with bounded attempts recover transient tool failures ($1-f^k$).
- MCP standardises the *wiring* to tools; it does not make tool use *safe* — that stays your job.

## Key takeaways

- **Success $= p^n$:** agent reliability compounds, so long trajectories punish any
  per-step unreliability (verified 0.95¹⁰≈0.60).
- **Retries rescue flaky steps** ($1-f^k$) — the cheapest lever — *if* the tool is
  idempotent (verified).
- **Raising per-step reliability compounds in your favour;** making steps retryable is
  often the highest-leverage design choice (demo).
- **MCP standardises tool descriptions/invocation** so the same tools work across agents
  and hosts.